# Scikit-Learn Notebook

In [43]:
!pip3 install scikit-learn

In [44]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.datasets import make_classification, make_regression

## 1) Estimator API workflow

In [45]:
X_cls, y_cls = make_classification(
    n_samples=300, n_features=4, n_informative=3, n_redundant=0, random_state=42
)
X_cls.shape, y_cls.shape

((300, 4), (300,))

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape

((240, 4), (60, 4), (240,))

In [47]:
clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=500)

In [48]:
preds = clf.predict(X_test)
accuracy_score(y_test, preds)

0.9

## 2) Preprocessing: scaling

In [49]:
X_train.mean(axis=0)

array([-0.51795232,  0.11246622,  0.06891482,  0.51664903])

In [50]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [51]:
X_test_scaled = scaler.transform(X_test)
X_train_scaled.mean(axis=0)

array([0.46210489, 0.57945245, 0.55549898, 0.56072348])

## 3) Regression example

In [52]:
X_reg, y_reg = make_regression(n_samples=200, n_features=3, noise=5.0, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)


In [53]:
reg = LinearRegression()
reg.fit(Xr_train, yr_train)

LinearRegression()

In [54]:
reg_preds = reg.predict(Xr_test)
mean_squared_error(yr_test, reg_preds) ** 0.5  # RMSE

5.75811877679597

## 4) Cross-validation

In [55]:
cv_scores = cross_val_score(LogisticRegression(max_iter=500), X_cls, y_cls, cv=5)
print(cv_scores)
cv_scores.mean()

[0.83333333 0.86666667 0.8        0.88333333 0.85      ]


np.float64(0.8466666666666667)

## 5) GridSearchCV: Hyperparameter Tuning

In [56]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid for LogisticRegression
param_grid = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['l2', 'l1'],
    'max_iter': [100, 500]
}

# Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=LogisticRegression(solver='liblinear'),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

# Fit the grid search (tests all combinations)
grid_search.fit(X_train, y_train)

# Display best parameters and score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

Best Parameters: {'C': 0.1, 'max_iter': 100, 'penalty': 'l1'}
Best CV Score: 0.8333


In [57]:
# Get best model and make predictions
best_model = grid_search.best_estimator_
best_preds = best_model.predict(X_test)
print(f"Test Accuracy with Best Model: {accuracy_score(y_test, best_preds):.4f}")

Test Accuracy with Best Model: 0.9167


In [18]:
# View all results as DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)
# Show top 5 parameter combinations by mean test score
results_df[['param_C', 'param_penalty', 'param_max_iter', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head()

,param_C,param_penalty,param_max_iter,mean_test_score
1,0.1,l1,100,0.833333
3,0.1,l1,500,0.833333
5,1.0,l1,100,0.829167
7,1.0,l1,500,0.829167
0,0.1,l2,100,0.820833
